<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/03_linguistic_structure_and_interpretability/dependency_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linguistic Structure: Dependency Parsing

This lab uses classical dependency parsing to pull Subject-Verb-Object facts out of text, the way early information extraction systems worked before deep learning. We then stress-test that approach until it breaks, first on toy sentences and then on real Helcon tickets. The point is to feel exactly where rule-based syntax stops working, which sets up why the next session moves to dense vector semantics for the Helcon Support Copilot.

In [ ]:
# Setup, run this cell first.
# It installs spaCy (the industry standard for classical NLP pipelines)
# and downloads the small English model used throughout the session.
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

# Repo access so the Helcon seed corpus (load_seed) is importable.
# Locally the repo is already on disk; on Colab we clone it.
REPO = "https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW"
import os, sys
if not os.path.exists("NLP-MBDS-EN") and not os.path.exists("../system/system_io.py"):
    !git clone --depth 1 -q $REPO NLP-MBDS-EN
for cand in ("../system", "NLP-MBDS-EN/system", "system"):
    if os.path.exists(os.path.join(cand, "system_io.py")):
        sys.path.insert(0, cand); break
from system_io import load_seed

print("Setup complete.")

## Step 1: Visualizing linguistic structure
Before deep learning, we relied on parsing sentences into directed graphs (dependency trees) to understand grammatical relationships.

In [ ]:
SENTENCE = "The quick brown fox jumps over the lazy dog."
# try: "The old man the boat."  (a garden path: watch the parser commit to a reading)
#      "I saw the man with the telescope."  (who has the telescope?)
#      "Scientists count whales from space."  (a real headline, same ambiguity)

import spacy
from spacy import displacy
from IPython.display import display, HTML

# Load the small English model and parse the sentence into a dependency tree.
nlp = spacy.load("en_core_web_sm")
doc = nlp(SENTENCE)

# Render the tree as an arc diagram (subject/object/modifier links between tokens).
html = displacy.render(doc, style="dep", jupyter=False, options={"distance": 100})
display(HTML(html))

The arc labels (`nsubj`, `dobj`, `prep`...) are the vocabulary of this whole session, and nobody is expected to know them by heart. The cell below prints the decoder ring for the sentence currently in `doc`. Change `SENTENCE` above, re-run both cells, and the table follows.

In [ ]:
# Print each token with its dependency label and a plain-English gloss,
# so the arc labels in the diagram above stop being cryptic.
print(f"{'TOKEN':<12} {'DEP':<10} MEANING")
print("-" * 60)
for tok in doc:
    print(f"{tok.text:<12} {tok.dep_:<10} {spacy.explain(tok.dep_) or 'sentence root'}")

## Step 2: The illusion of rule-based information extraction
Let's build a classic Subject-Verb-Object (SVO) extractor. This was the foundation of early knowledge graphs and question answering (QA) systems.

Before running: the corpus contains *"Mary gave John a book"*. Will the extracted triple mention John?

In [ ]:
def extract_svo_triples(text, nlp_model):
    doc = nlp_model(text)
    svo_triples = []

    for token in doc:
        # A token whose dependency contains "subj" is the subject of its head verb.
        if "subj" in token.dep_:
            subject = token
            verb = token.head
            # Walk the verb's children looking for an object to complete the triple.
            for obj in verb.children:
                if "obj" in obj.dep_:                  # direct/indirect object on the verb
                    svo_triples.append((subject, verb, obj))
                elif obj.dep_ == "prep":               # prepositional phrase: dig one level for its object
                    for pobj in obj.children:
                        if pobj.dep_ == "pobj":
                            svo_triples.append((subject, verb, pobj))
    return svo_triples

# Run it on clean, textbook-perfect sentences first, where the rule looks flawless.
sterile_text = "John bought a new car. Mary gave John a book. Alice traveled to Paris."
sterile_triples = extract_svo_triples(sterile_text, nlp)

print("--- Extracted Triples ---")
for triple in sterile_triples:
    print(f"Subject: {triple[0].text: <5} | Verb: {triple[1].text: <8} | Object: {triple[2].text}")

No John. *Mary gave John a book* came out as *(Mary, gave, book)*: the parser labels the recipient `dative`, our rule only collects labels containing `obj`, and the recipient was silently dropped. No error, no warning, the information simply never made it into the triple. Remember this every time an extraction pipeline "works".

## Step 3: Building a simple QA system
We can use these triples to answer basic questions by matching the verb.

In [ ]:
def simple_qa(question, svo_triples, nlp_model):
    question_doc = nlp_model(question)
    question_verb = None

    # The question's main verb is our only matching key into the triple store.
    for token in question_doc:
        if token.pos_ == "VERB":
            question_verb = token
            break

    if question_verb is not None:
        for triple in svo_triples:
            subject, verb, obj = triple
            # Match on lemma so tense does not matter (bought -> buy).
            if verb.lemma_ == question_verb.lemma_:
                return f"{subject.text} {verb.text} {obj.text}"

    return "System Failure: Cannot determine answer."

print("Question: Who bought a car?")
print("Answer:  ", simple_qa("Who bought a car?", sterile_triples, nlp))

## Step 4: The stress test
The system above looks like artificial intelligence. **It is not.** It is a rigid topological rule.

Let's see what happens when we introduce standard linguistic variance (synonyms, passive voice, complex clauses).

In [ ]:
# The real world does not speak in perfect SVO structures.
real_world_text = "A brand new car was purchased by John. The book, which was heavy, was given to John by Mary."
real_world_triples = extract_svo_triples(real_world_text, nlp)

print("--- QA Stress Test ---")
questions = [
    "Who bought a car?",           # fails: synonym (purchased vs bought)
    "Who purchased the car?",      # fails: passive voice makes 'car' the grammatical subject
    "Who gave the book to John?"   # fails: passive voice makes 'book' the grammatical subject
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {simple_qa(q, real_world_triples, nlp)}")

# Print the raw triples so the failure is visible: the extractor found none.
print("\n--- Why did it fail? Look at the Triples ---")
if not real_world_triples:
    print("(nothing: the extractor found no triples at all in the passive text)")
for triple in real_world_triples:
    print(f"Parsed -> Subject: {triple[0].text: <4} | Verb: {triple[1].text: <9} | Object: {triple[2].text}")

All three questions died, and the triple list at the bottom is the autopsy: the extractor produced nothing at all. In the passive, the parser labels *car* as `nsubjpass` (grammatically a subject), so our rule accepts it and then searches the verb's children for an object that does not exist: the only other nouns hide inside the *by John* and *to John* phrases, under labels (`agent`, `dative`) that our rules never visit. And the synonym question would fail even with perfect extraction, because *purchased* and *bought* are different lemmas and a tree carries no notion that they mean the same thing.

One more stressor before we fix anything, this one subtler than the passive. Before running: the text below states that John **never** bought a car. What will the QA answer?

In [ ]:
# Negation: the assertion is reversed, but a neg modifier is not on a subject/object arc.
negated_text = "John never bought a car."
negated_triples = extract_svo_triples(negated_text, nlp)

print("--- Extracted Triples ---")
for s, v, o in negated_triples:
    print(f"Subject: {s.text: <5} | Verb: {v.text: <8} | Object: {o.text}")

print("\nQ: Who bought a car?")
print("A: ", simple_qa("Who bought a car?", negated_triples, nlp))

The system cheerfully answers that John bought a car, the text asserts the exact opposite. The word *never* hangs off the verb as a `neg` modifier, but our traversal only walks subject and object arcs, so the subject-verb-object skeleton is identical with or without the negation. Negation is invisible to tree topology unless someone writes yet another rule for it. Notice the pattern forming: every failure so far has the same prescription, one more rule.

## Step 5: Fixing the passive
The passive failures, at least, look fixable, and the parser already gives us everything we need. `nsubjpass` marks the thing acted upon, and the true actor hides inside the `agent` phrase (*by John*). So the new rule is: treat the passive subject as the OBJECT, and recover the real subject by walking the verb's children for the agent.

In [ ]:
def extract_svo_triples_v2(text, nlp_model):
    doc = nlp_model(text)
    triples = []
    for token in doc:
        if token.dep_ == "nsubjpass":              # passive: grammatical subject = real object
            verb = token.head
            for child in verb.children:
                if child.dep_ == "agent":          # the "by ..." phrase carries the real actor
                    for pobj in child.children:
                        if pobj.dep_ == "pobj":    # its object = the real subject
                            triples.append((pobj, verb, token))
        elif "subj" in token.dep_:                 # active voice: same rule as before
            verb = token.head
            for obj in verb.children:
                if "obj" in obj.dep_:
                    triples.append((token, verb, obj))
    return triples

real_world_triples_v2 = extract_svo_triples_v2(real_world_text, nlp)
print("--- Extracted Triples (v2) ---")
for s, v, o in real_world_triples_v2:
    print(f"Subject: {s.text: <5} | Verb: {v.text: <9} | Object: {o.text}")

print("\n--- QA Stress Test, round 2 ---")
for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {simple_qa(q, real_world_triples_v2, nlp)}")

Two of the three questions now pass. The survivor is the synonym: the text says *purchased*, the question says *bought*, and no rule about tree shapes will ever make those two lemmas touch. The tree encodes structure, not meaning. That residual failure is the bridge to the next session, where dense vector semantics put *purchase* and *buy* next to each other in space.

Before celebrating the passive fix, though, one last sentence. Before running: v2 "understands" passives now, so what does it extract from *"John was given the book by Mary."*?

In [ ]:
# The v2 fix was tuned for "A car was purchased". English also promotes the
# recipient to passive subject, and our new rule reads that case incorrectly.
tricky_text = "John was given the book by Mary."
tricky_triples = extract_svo_triples_v2(tricky_text, nlp)

print("--- Extracted Triples (v2) ---")
for s, v, o in tricky_triples:
    print(f"Subject: {s.text: <5} | Verb: {v.text: <8} | Object: {o.text}")

According to v2, what Mary gave was... John. The rule "the passive subject is the thing acted upon" was written for *A car was purchased*, but English also promotes the **recipient** to passive subject (*John was given the book*), and our brand-new rule misreads it with total confidence: the book, the thing actually given, does not even appear in the triple. It gets worse: the parser's own labels wobble. Change *the book* to *a book* and the current small model stops labeling *by* as `agent`, so v2 silently extracts nothing at all. We fixed one construction and immediately broke another. This is the whack-a-mole dynamic of rule-based NLP: every rule spawns two new exceptions.

## Step 6: Now on our own data, Helcon support tickets

Everything above broke on sentences we hand-picked to break it. Fair to ask whether that is a contrived problem. So let's point the *unchanged* extractor and QA at real Helcon support tickets, a curated set of hard cases pulled straight from the 748-ticket corpus. Before running: these are the exact failure modes from the toy sentences (passive, negation, synonym), now in production customer language.

In [ ]:
import pandas as pd

# Load the curated probe set and the full ticket corpus via the shared SDK (system_io).
probes = load_seed("probes")
tickets = pd.DataFrame(load_seed("tickets"))

# A little color on what we are working with.
print(f"Helcon corpus: {len(tickets)} support tickets across "
      f"{tickets['category'].nunique()} triage categories.")
print(tickets["category"].value_counts().to_string())

# Pull one representative ticket from each failure group.
passive = probes["passive_voice"][0]   # HLC-P001
negation = probes["negation"][0]       # HLC-P004
synonym = probes["synonym"][0]         # HLC-P007

print("\nThe three probe tickets we will run through the toy system:")
for p in (passive, negation, synonym):
    print(f"  [{p['id']}] {p['text']}")

In [ ]:
# The extractor and QA below are the ORIGINAL toy functions, unchanged.
# We only change the input: real customer tickets instead of toy sentences.

def report(ticket, question, means):
    triples = extract_svo_triples(ticket["text"], nlp)
    print(f"[{ticket['id']}] {ticket['text']}")
    if triples:
        for s, v, o in triples:
            print(f"   extractor saw -> ({s.text}, {v.text}, {o.text})")
    else:
        print("   extractor saw -> (no triples at all)")
    print(f"   QA: {question}")
    print(f"   ->  {simple_qa(question, triples, nlp)}")
    print(f"   but the ticket actually means: {means}\n")

print("--- The same three failures, now on production tickets ---\n")

# Passive: the refund clause has no surviving actor; the only triple is the
# unrelated 'nobody contacted me' half.
report(passive, "Whose refund was processed?",
       "a refund was promised and never happened, a BILLING escalation")

# Negation: 'NOT' hangs off the verb as a neg modifier the traversal never visits,
# so QA reads the customer as ASSERTING the charge.
report(negation, "Who authorized this charge?",
       "the customer is DISPUTING the charge, the exact opposite")

# Synonym: 'reimburse' / 'double-billed' never share a lemma with 'refund',
# so the refund intent simply never registers.
report(synonym, "Who wants a refund?",
       "a refund request for a double charge, missed entirely")

These are real Helcon support tickets, not sentences we invented to embarrass the parser. The passive ticket loses the refund clause, the negation ticket is read as the customer *agreeing* to the charge, and the synonym ticket's refund request never registers: the same three failures that broke our toy sentences, now breaking on the language customers actually write. No extra rule survives contact with this variety. That is precisely why the next session abandons tree topology for dense semantics, where *reimburse* and *refund* finally sit next to each other in space.

## Takeaway

We did fix the passive (a dozen lines, one extra rule) and it survived exactly one sentence before the ditransitive passive broke it again. Negation never registered at all, and the synonym failure cannot be fixed by any structural rule, because *purchased* vs *bought* is a fact about meaning, not about trees. Rule-based extraction does not fail because the rules are badly written; it fails because language has more constructions than any rulebook can enumerate, and every patch opens new holes. What the system lacks is not another rule but a representation of meaning: dense vector semantics, next session.